# ML Week 17: Object Detection

Last week we saw Image Classification, which applies to images in which there is a single object. What if we have many objects in an image, potentially small and not-centred?

In computer vision, this task is known as "Object Detection".

We ultimately wish to return an updated version of the image with a "bounding box" and label for each object. For example:

<div>
<img src="./YOLO_example.png" width="600"/>
</div>

This is a very active area of research that is changing lightning fast. In this notebook we explore some of the current best models.

### YOLO

One of the earliest implementations was YOLO (You Only Look Once), first released in 2015 by Joseph Redmon:

https://arxiv.org/abs/1506.02640

Homepage: https://pjreddie.com/

YOLO has since become one of the gold standards for Object Detection. It is managed by an organization called Ultralytics and updated regularly. The most recent version is YOLO26 (released Feb 2026, paper: https://arxiv.org/pdf/2602.14582).

Out-of-the box YOLO is exceptionally sipmle to use!

It is trained on the COCO Dataset (Common Objects in Context) which contains 80 classes.

Let's give it a try!

In [13]:
from ultralytics import YOLO

In [23]:
# Load the latest model (pre-trained)
# Note the "n" after 26 means "nano". Each YOLO version comes in 5 sizes: nano (n), small (s), medium (m), large (l) and extra large (x).
# You should choose whatever your computer can handle (but note that training is really heavy)

model = YOLO("yolo26n.pt")

# Display model information (optional)
model.info()

YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.1 GFLOPs


(260, 2572280, 0, 6.1192448)

In [24]:
# Apply the model to an image

results = model.predict("Coolidge.png")


image 1/1 c:\Users\re72fam\Documents\winter 25-26\MLADCH2526\week17\Coolidge.png: 544x640 3 dogs, 2 bottles, 1 cup, 1 chair, 48.1ms
Speed: 2.4ms preprocess, 48.1ms inference, 0.2ms postprocess per image at shape (1, 3, 544, 640)


In [25]:
results[0].boxes

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([16., 39., 16., 41., 39., 16., 56.])
conf: tensor([0.8405, 0.7973, 0.7379, 0.6358, 0.6028, 0.5824, 0.2892])
data: tensor([[0.0000e+00, 4.3392e+02, 8.2473e+02, 1.5350e+03, 8.4050e-01, 1.6000e+01],
        [3.4970e+02, 1.1850e+03, 5.2322e+02, 1.6092e+03, 7.9730e-01, 3.9000e+01],
        [7.4429e+02, 6.7992e+02, 1.2573e+03, 1.1434e+03, 7.3792e-01, 1.6000e+01],
        [6.4092e+02, 1.4123e+03, 7.4531e+02, 1.5610e+03, 6.3580e-01, 4.1000e+01],
        [1.2710e+03, 7.7353e+02, 1.3903e+03, 1.1159e+03, 6.0276e-01, 3.9000e+01],
        [1.0460e+03, 6.8600e+02, 1.9990e+03, 1.5105e+03, 5.8241e-01, 1.6000e+01],
        [6.9639e+02, 4.6134e+02, 1.2884e+03, 1.1310e+03, 2.8919e-01, 5.6000e+01]])
id: None
is_track: False
orig_shape: (1660, 2000)
shape: torch.Size([7, 6])
xywh: tensor([[ 412.3637,  984.4844,  824.7274, 1101.1202],
        [ 436.4603, 1397.0923,  173.5120,  424.2528],
        [1000.7719,  911.6843,  512.9738,  463.5236

In [26]:
# Check the results

boxes = results[0].boxes

for box in boxes:
    print("Object type:", box.cls)
    print("Coordinates:", box.xyxy)
    print("Probability:", box.conf)

Object type: tensor([16.])
Coordinates: tensor([[   0.0000,  433.9243,  824.7274, 1535.0446]])
Probability: tensor([0.8405])
Object type: tensor([39.])
Coordinates: tensor([[ 349.7043, 1184.9658,  523.2163, 1609.2186]])
Probability: tensor([0.7973])
Object type: tensor([16.])
Coordinates: tensor([[ 744.2850,  679.9225, 1257.2588, 1143.4460]])
Probability: tensor([0.7379])
Object type: tensor([41.])
Coordinates: tensor([[ 640.9232, 1412.2703,  745.3071, 1560.9550]])
Probability: tensor([0.6358])
Object type: tensor([39.])
Coordinates: tensor([[1271.0261,  773.5274, 1390.2549, 1115.8777]])
Probability: tensor([0.6028])
Object type: tensor([16.])
Coordinates: tensor([[1045.9576,  686.0023, 1999.0040, 1510.5353]])
Probability: tensor([0.5824])
Object type: tensor([56.])
Coordinates: tensor([[ 696.3889,  461.3414, 1288.3728, 1130.9932]])
Probability: tensor([0.2892])


In [27]:
# Process results list and show the final image with bounding boxes and labels

for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen

### A Note About Transformers

In 2016 at Google a team of researchers sat for lunch and started chatting about Natural Language Processing (NLP).

Their conversation would go on to transform modern AI through a landmark paper called "Attention is All You Need" (https://arxiv.org/abs/1706.03762).

The paper instroduces a "Transformer" architecture which consists of MLPs linked through multiple "Attention" blocks. These blocks allow embedding vectors to "chat" with each other, learning context across long embeddings.

Initially this architecture was entirely for NLP, but soon it was discovered that Attention blocks also work great in Computer Vision by allowing different parts of the image to communicate contextual information.

YOLOv26, the model we used above, contains a single attention block called C2PSA (Cross Stage Partial with Spatial Attention). It was introduced in YOLOv10, and significantly improved performance.

This is a super duper quick intro to Transformers (and Attention). For an in-depth look into them, I strongly recommend this video series:

https://www.youtube.com/watch?v=wjZofJX0v4M

### DETR

In 2021 DETR (Detection Transformer) was introduced by Facebook. As opposed to YOLOv10 (introduced in 2024) which contained a single attention block, DETR contained multiple attention blocks (more like the transformers used for NLP). This made it very powerful at understanding complex image context.

In 2023 researchers at Baidu released the Real Time Detection Transformer (RT-DETR). Speed ups make it able to work efficiently on video.

RT-DETR has since been included in the Ultralytics library, and can be used exactly as with YOLO versions.

**Note**: Baidu has since released RT-DETRv2, but I don't believe it's been yet included the ultralytics library.

In [20]:
from ultralytics import RTDETR

# Load a COCO-pretrained RT-DETR-l model
model = RTDETR("rtdetr-l.pt")

# Display model information (optional)
model.info()

# Run inference with the RT-DETR-l model on the 'bus.jpg' image
results = model("Coolidge.png")

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs

image 1/1 c:\Users\re72fam\Documents\winter 25-26\MLADCH2526\week17\Coolidge.png: 640x640 4 dogs, 2 bottles, 2 cups, 1 bowl, 2 chairs, 1 book, 432.5ms
Speed: 2.8ms preprocess, 432.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)


In [55]:
results[0].boxes

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([15., 57., 15., 65., 65., 59., 59.])
conf: tensor([0.9595, 0.9552, 0.9503, 0.9436, 0.9237, 0.3578, 0.2576])
data: tensor([[ 3.4479e+02,  2.4742e+01,  6.4014e+02,  3.7097e+02,  9.5948e-01,  1.5000e+01],
        [ 3.4660e-01, -7.9825e-01,  6.4031e+02,  4.7854e+02,  9.5523e-01,  5.7000e+01],
        [ 1.4613e+01,  5.5131e+01,  3.1814e+02,  4.7202e+02,  9.5035e-01,  1.5000e+01],
        [ 4.0033e+01,  7.3111e+01,  1.7570e+02,  1.1773e+02,  9.4360e-01,  6.5000e+01],
        [ 3.3389e+02,  7.6533e+01,  3.7042e+02,  1.8807e+02,  9.2366e-01,  6.5000e+01],
        [ 8.4469e-01, -9.3555e-03,  6.4084e+02,  4.7730e+02,  3.5778e-01,  5.9000e+01],
        [ 5.1472e-01, -1.1451e+00,  6.4050e+02,  4.7884e+02,  2.5764e-01,  5.9000e+01]])
id: None
is_track: False
orig_shape: (480, 640)
shape: torch.Size([7, 6])
xywh: tensor([[492.4657, 197.8570, 295.3487, 346.2309],
        [320.3282, 238.8723, 639.9631, 479.3411],
        [166.3762, 

In [22]:
# Process results list
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen

We see that this model picks up on a few more objects than YOLO.

### Optional: Object Detection with Hugging Face 🤗

A competing standard to Ultralytics is Hugging Face, which contains an incredibly in depth library of nearly all modern transformer architectures.

However, using the models "out-of-box" is not as straightforward. Notably, I was not able to run them without first creating what is called a "virtual environment" within which the dependencies had to be installed.

For the interested (and also partially for myself) I include a working example with Hugging Face DETR Here.

But in general I would suggest skipping over this to the next section.

In [28]:
from transformers import pipeline
import torch
from PIL import Image
import requests
import matplotlib.pyplot as plt

W0527 02:22:17.272000 48572 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [29]:
pipeline = pipeline(
    "object-detection",
    model="facebook/detr-resnet-50"
)

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
img = Image.open(requests.get(url, stream=True).raw)
outputs = pipeline(url)

In [31]:
outputs

[{'score': 0.9982200264930725,
  'label': 'remote',
  'box': {'xmin': 40, 'ymin': 70, 'xmax': 175, 'ymax': 117}},
 {'score': 0.995998740196228,
  'label': 'remote',
  'box': {'xmin': 333, 'ymin': 72, 'xmax': 368, 'ymax': 187}},
 {'score': 0.9954814910888672,
  'label': 'couch',
  'box': {'xmin': 0, 'ymin': 1, 'xmax': 639, 'ymax': 473}},
 {'score': 0.9988006353378296,
  'label': 'cat',
  'box': {'xmin': 13, 'ymin': 52, 'xmax': 314, 'ymax': 470}},
 {'score': 0.9986791014671326,
  'label': 'cat',
  'box': {'xmin': 345, 'ymin': 23, 'xmax': 640, 'ymax': 368}}]

In [46]:
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

def plot_results(img, outputs):
    plt.figure(figsize=(16,10))
    plt.imshow(img)
    ax = plt.gca()
    colors = COLORS * 100
    color_dict = {}
    for output in outputs:
        prob = output['score']
        label = output['label']
        if label not in color_dict.keys():
            color_dict[label] = colors.pop()
        xmin, ymin, xmax, ymax = list(output['box'].values())
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, color=color_dict[label], fill=False, linewidth=3))
        text = f'{label}: {prob:0.2f}'
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor='yellow', alpha=0.5))
    plt.axis('off')
    plt.show()

In [47]:
plot_results(img, outputs)

<Figure size 1600x1000 with 1 Axes>

### Transfer Learning in Object Detection

In [12]:
## Train the model on the COCO8 example dataset for 100 epochs
results = model.train(data="coco8.yaml", epochs=100, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.55 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.5 torch-2.11.0+cpu CPU (13th Gen Intel Core i9-13900H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-5, nbs=64, 